# PyTorch Playground demo\n
\n
`nn.Sequential` の訓練は Python/PyTorch で行い、記録したスナップショットだけを HTML ビューアで再生するデモです。

In [1]:
import numpy as np
import torch
from torch import nn
from IPython.display import HTML

from nitic_playground import create_playground_recorder

np.random.seed(1)
torch.manual_seed(1)

max_epochs = 200
n = 180
X = np.random.randn(n, 2).astype("float32")
y = ((X[:, 0] ** 2 + X[:, 1] ** 2) > 1.0).astype("float32")

model = nn.Sequential(
    nn.Linear(2, 8),
    nn.Tanh(),
    nn.Linear(8, 1),
)
optimizer = torch.optim.SGD(model.parameters(), lr=0.08)
criterion = nn.BCEWithLogitsLoss()
recorder = create_playground_recorder(
    model,
    X,
    y,
    problem="classification",
    title="PyTorch Playground Demo: Circle Classification",
    grid_size=56,
)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)
for epoch in range(max_epochs):
    optimizer.zero_grad()
    logits = model(X_t)[:, 0]
    loss = criterion(logits, y_t)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        with torch.no_grad():
            acc = ((torch.sigmoid(model(X_t)[:, 0]) >= 0.5) == y_t.bool()).float().mean().item()
        recorder.capture(step=epoch, loss=float(loss.item()), metric=float(acc))

In [2]:
HTML(recorder.to_html())

frames: 13
first loss: 0.6973
last loss: 0.6716
last accuracy: 54.4%


In [5]:
print(f"frames: {len(recorder.frames)}")
print(f"first loss: {recorder.frames[0]['loss']:.4f}")
print(f"last loss: {recorder.frames[-1]['loss']:.4f}")
print(f"last accuracy: {recorder.frames[-1]['metric'] * 100:.1f}%")

frames: 13
first loss: 0.6973
last loss: 0.6716
last accuracy: 54.4%
